# Fase 2.3 — Modelo TTM (Tiny Time Mixers — IBM Granite)

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook evaluates the IBM Granite Tiny Time Mixers foundation model
(`ibm/granite-timeseries-ttm-r2`) in two modes:

1. **Zero-shot** — load the pre-trained model, run inference with no fine-tuning
2. **Few-shot** — fine-tune on 5% of training data, then evaluate on the test set

The model context length is **512 hours** → forecast horizon **24 hours**.

### Installation (run once in your venv)
```bash
pip install "granite-tsfm[notebooks]>=0.2" transformers>=4.40 datasets>=2.14 accelerate>=0.27
```

In [ ]:
import sys, os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch

sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

os.makedirs('data', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# granite-tsfm imports
try:
    from tsfm_public.models.tinytimemixer import (
        TinyTimeMixerForPrediction,
        TinyTimeMixerConfig,
    )
    from tsfm_public import TinyTimeMixerForPrediction  # noqa (alias)
    from tsfm_public.toolkit.dataset import ForecastDFDataset
    from tsfm_public.toolkit.time_series_forecasting_pipeline import TimeSeriesForecastingPipeline
    from tsfm_public.toolkit.trainer import Trainer, TrainingArguments
    print('granite-tsfm imported successfully')
except ImportError as e:
    print(f'Import error: {e}')
    print('Install with: pip install "granite-tsfm[notebooks]"')
    raise

## 1 · Load and prepare data

In [ ]:
df_train = pd.read_csv('data/features_train.csv', parse_dates=['datetime'])
df_val   = pd.read_csv('data/features_val.csv',   parse_dates=['datetime'])
df_test  = pd.read_csv('data/features_test.csv',  parse_dates=['datetime'])

# TTM model parameters
CONTEXT_LEN = 512    # pre-trained context window
HORIZON     = 24     # 24h forecast
TARGET_COL  = 'demand_mw'

# Conditional (exogenous) columns — drop datetime, keep numeric
COND_COLS = [c for c in df_train.columns
             if c not in ['datetime', TARGET_COL]]

print(f'Target   : {TARGET_COL}')
print(f'Conditional columns: {len(COND_COLS)}')
print(f'Train : {len(df_train):,}  Val : {len(df_val):,}  Test : {len(df_test):,}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# TTM requires StandardScaler per channel (target + conditional)
all_cols = [TARGET_COL] + COND_COLS

scaler = StandardScaler()
scaler.fit(df_train[all_cols])

def scale_df(df, cols=all_cols):
    out = df.copy()
    out[cols] = scaler.transform(df[cols])
    return out

df_train_sc = scale_df(df_train)
df_val_sc   = scale_df(df_val)
df_test_sc  = scale_df(df_test)

# Concatenated history for zero-shot inference (train + val as context)
df_history = pd.concat([df_train_sc, df_val_sc], ignore_index=True)
print(f'History length for zero-shot: {len(df_history):,} rows')

## 2 · Load pre-trained TTM model

In [ ]:
TTM_MODEL_ID = 'ibm/granite-timeseries-ttm-r2'

print(f'Loading {TTM_MODEL_ID}…')
zs_model = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_ID,
    prediction_filter_length = HORIZON,
)
zs_model = zs_model.to(DEVICE)
zs_model.eval()

config = zs_model.config
print(f'Context length : {config.context_length}')
print(f'Forecast length: {config.prediction_length}')
print(f'Parameters     : {sum(p.numel() for p in zs_model.parameters()):,}')

## 3 · Build ForecastDFDatasets

In [ ]:
def make_dataset(df, context_len=CONTEXT_LEN, horizon=HORIZON, stride=1):
    """Wrap a DataFrame in ForecastDFDataset expected by granite-tsfm."""
    return ForecastDFDataset(
        df,
        id_columns           = [],
        timestamp_column     = 'datetime',
        target_columns       = [TARGET_COL],
        conditional_columns  = COND_COLS,
        context_length       = context_len,
        prediction_length    = horizon,
        stride               = stride,
    )

# Zero-shot test dataset: use last CONTEXT_LEN rows of history as prefix,
# then test data — ensures every prediction window falls within the test period.
df_for_zs_test = pd.concat([
    df_history.tail(CONTEXT_LEN).reset_index(drop=True),
    df_test_sc,
], ignore_index=True)

# stride=HORIZON gives non-overlapping daily predictions
test_dataset_zs = make_dataset(df_for_zs_test, stride=HORIZON)
print(f'Zero-shot test dataset: {len(test_dataset_zs)} windows')

# Few-shot fine-tuning dataset (5% of train)
n_fewshot   = max(CONTEXT_LEN + HORIZON, int(0.05 * len(df_train_sc)))
df_fewshot  = df_train_sc.tail(n_fewshot).reset_index(drop=True)
fs_train_ds = make_dataset(df_fewshot, stride=1)
fs_val_ds   = make_dataset(df_val_sc,  stride=HORIZON)
print(f'Few-shot train dataset  : {len(fs_train_ds)} windows  (from {n_fewshot} rows = ~5% of train)')

## 4 · Zero-shot evaluation

In [ ]:
from torch.utils.data import DataLoader

t0  = time.time()
loader = DataLoader(test_dataset_zs, batch_size=64, shuffle=False)

preds_zs, trues_zs = [], []

with torch.no_grad():
    for batch in loader:
        past_values = batch['past_values'].to(DEVICE)
        outputs     = zs_model(past_values=past_values)
        preds_zs.append(outputs.prediction_outputs.cpu().numpy())
        trues_zs.append(batch['future_values'].numpy())

inference_time_zs = time.time() - t0

preds_zs = np.concatenate(preds_zs, axis=0)   # (n_windows, horizon, 1)
trues_zs = np.concatenate(trues_zs, axis=0)

# Extract target channel (index 0) and inverse-transform
target_mean = scaler.mean_[0]
target_std  = scaler.scale_[0]

y_pred_zs = preds_zs[:, :, 0].ravel() * target_std + target_mean
y_true_zs = trues_zs[:, :, 0].ravel() * target_std + target_mean

print(f'Inference time (zero-shot): {inference_time_zs:.2f}s')
print(f'Forecast points: {len(y_pred_zs):,}')

In [ ]:
metrics_zs = du.compute_metrics(y_true_zs, y_pred_zs, label='TTM zero-shot')
metrics_zs['train_s']     = 0.0
metrics_zs['inference_s'] = inference_time_zs

## 5 · Few-shot fine-tuning (5% of training data)

In [ ]:
# Clone the pre-trained model for fine-tuning
fs_model = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL_ID,
    prediction_filter_length = HORIZON,
)

# Enable channel-mixing in the decoder (captures cross-variable correlations)
fs_model.config.decoder_channel_mixing = True

training_args = TrainingArguments(
    output_dir             = 'saved_models/ttm_fewshot',
    num_train_epochs       = 20,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size  = 64,
    learning_rate          = 1e-4,
    lr_scheduler_type      = 'cosine',
    warmup_ratio           = 0.1,
    evaluation_strategy    = 'epoch',
    save_strategy          = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'eval_loss',
    logging_dir            = 'saved_models/ttm_fewshot/logs',
    logging_steps          = 10,
    report_to              = 'none',
    seed                   = 42,
)

trainer = Trainer(
    model        = fs_model,
    args         = training_args,
    train_dataset= fs_train_ds,
    eval_dataset = fs_val_ds,
)

print(f'Fine-tuning on {len(fs_train_ds)} windows ({n_fewshot} rows)…')
t0 = time.time()
trainer.train()
finetune_time = time.time() - t0
print(f'Fine-tuning completed in {finetune_time:.1f}s  ({finetune_time/60:.1f} min)')

In [ ]:
# Plot fine-tuning loss
log_history = trainer.state.log_history
train_losses = [e['loss'] for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_losses  = [e['eval_loss'] for e in log_history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train loss')
if eval_losses:
    eval_x = np.linspace(0, len(train_losses)-1, len(eval_losses))
    ax.plot(eval_x, eval_losses, label='Val loss', linestyle='--')
ax.set_title('TTM few-shot fine-tuning loss')
ax.set_xlabel('Step')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_ttm_finetune.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 · Few-shot inference on test set

In [ ]:
fs_model.eval()
fs_model = fs_model.to(DEVICE)

# Re-use the same test dataset built for zero-shot
fs_loader = DataLoader(test_dataset_zs, batch_size=64, shuffle=False)

t0 = time.time()
preds_fs, trues_fs = [], []

with torch.no_grad():
    for batch in fs_loader:
        past_values = batch['past_values'].to(DEVICE)
        outputs     = fs_model(past_values=past_values)
        preds_fs.append(outputs.prediction_outputs.cpu().numpy())
        trues_fs.append(batch['future_values'].numpy())

inference_time_fs = time.time() - t0

preds_fs = np.concatenate(preds_fs, axis=0)
trues_fs = np.concatenate(trues_fs, axis=0)

y_pred_fs = preds_fs[:, :, 0].ravel() * target_std + target_mean
y_true_fs = trues_fs[:, :, 0].ravel() * target_std + target_mean

print(f'Inference time (few-shot): {inference_time_fs:.2f}s')
print(f'Forecast points: {len(y_pred_fs):,}')

In [ ]:
metrics_fs = du.compute_metrics(y_true_fs, y_pred_fs, label='TTM few-shot')
metrics_fs['train_s']     = finetune_time
metrics_fs['inference_s'] = inference_time_fs

## 7 · Plots

In [ ]:
# Build datetime array for test predictions
n_windows  = len(preds_zs)
dates_test = df_for_zs_test['datetime'].values

dates_zs = np.concatenate([
    dates_test[CONTEXT_LEN + i * HORIZON : CONTEXT_LEN + i * HORIZON + HORIZON]
    for i in range(n_windows)
])
dates_zs = pd.to_datetime(dates_zs)

In [ ]:
du.plot_predictions(
    y_true_zs[:168], y_pred_zs[:168],
    title     = 'TTM zero-shot — first week of test set',
    dates     = dates_zs[:168],
    save_path = 'data/fig_ttm_zeroshot_week.png',
)

In [ ]:
du.plot_predictions(
    y_true_fs[:168], y_pred_fs[:168],
    title     = 'TTM few-shot — first week of test set',
    dates     = dates_zs[:168],
    save_path = 'data/fig_ttm_fewshot_week.png',
)

In [ ]:
# Side-by-side zero-shot vs few-shot (one representative week)
idx = slice(0, 168)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, y_pred, label, color in [
    (axes[0], y_pred_zs[idx], 'Zero-shot', '#ff7f0e'),
    (axes[1], y_pred_fs[idx], 'Few-shot',  '#2ca02c'),
]:
    ax.plot(dates_zs[idx], y_true_zs[idx], label='Actual',  color='#1f77b4', linewidth=1.2)
    ax.plot(dates_zs[idx], y_pred,          label=label,   color=color, linewidth=1.2, linestyle='--')
    ax.set_title(f'TTM {label}')
    ax.set_ylabel('Demand (MW)')
    ax.legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.tight_layout()
plt.savefig('data/fig_ttm_comparison_week.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Save predictions and metrics

In [ ]:
# Zero-shot
pd.DataFrame({'datetime': dates_zs, 'y_true': y_true_zs, 'y_pred': y_pred_zs}).to_csv(
    'data/predictions_ttm_zeroshot.csv', index=False
)
with open('data/metrics_ttm_zeroshot.json', 'w') as f:
    json.dump(metrics_zs, f, indent=2)

# Few-shot
pd.DataFrame({'datetime': dates_zs, 'y_true': y_true_fs, 'y_pred': y_pred_fs}).to_csv(
    'data/predictions_ttm_fewshot.csv', index=False
)
with open('data/metrics_ttm_fewshot.json', 'w') as f:
    json.dump(metrics_fs, f, indent=2)

print('Saved predictions and metrics for both TTM modes.')
print('\nZero-shot metrics:', metrics_zs)
print('Few-shot  metrics:', metrics_fs)

## Summary

| Mode | MAPE (%) | RMSE (MW) | MAE (MW) | Fine-tune time |
|------|----------|-----------|----------|----------------|
| Zero-shot | … | … | … | 0 s |
| Few-shot  | … | … | … | … s |

Next step → `fase3_validacion_y_comparativa.ipynb`